# Word-by-word glossing pilot (ICL, Bolmo 1B)

Compares two answer formats for in-context learning on the **same dev sentences with the same 21 retrieved examples**:
- **line** (current): `Gloss: way-AD.ESS mountain-SUPER go-PST`
- **word** (new): one line per transliteration word, `way = way-AD.ESS`, … (`icl/word_format.py`)

Why: our models often add or drop a word in the gloss, which shifts every later gloss and makes positional accuracy collapse. For scoring, the word lines are joined back into one gloss line, so both formats are scored the same way.

Use **Tsez** (all 445 test / 1,177 pool sentences are word-aligned) or Jejueo (91%). Not Ainu (only 34% aligned).

Before running: **Runtime → Change runtime type → T4 GPU** (float32, like the other ICL runs). `MODE = "test_word"` (default) runs the word format on the whole test split (Tsez: about 45 min on an L4) and compares it with the existing line-format test run; `MODE = "dev_pilot"` runs both formats on 50 dev rows.

## 1. Google Drive (results are saved there)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2. Get the code and install

If the last line asks you to restart: **Runtime → Restart session**, then run from section 1 again.

In [ ]:
import os
REPO = "/content/Korean-IGT-Bolmo"
BRANCH = "soyeong/word-by-word"   # until it is merged into main
if not os.path.isdir(REPO):
    !git clone -q -b {BRANCH} https://github.com/JingyiShao-Gina/Korean-IGT-Bolmo.git {REPO}
else:
    !git -C {REPO} fetch -q origin {BRANCH} && git -C {REPO} checkout -q {BRANCH} && git -C {REPO} reset -q --hard origin/{BRANCH}
os.chdir(REPO)
!git log --oneline -1
!pip install -q "transformers==4.57.3" "xlstm==2.0.4" datasets pandas accelerate sacrebleu
import transformers
print("transformers", transformers.__version__)
if transformers.__version__ != "4.57.3":
    print(">>> Restart the session (Runtime -> Restart session), then run the notebook again from the top.")

## 3. Settings

In [ ]:
LANG = "tsez"           # @param ["tsez", "jejueo"]
MODE = "test_word"      # @param ["test_word", "dev_pilot"]
# test_word: word format on the whole test split; compare with the existing line-format test run
#            (results/icl/<lang>/bolmo-1b_all_in_one_icl21_test.csv, e.g. Theo's Tsez run)
# dev_pilot: both formats on the first N dev rows
N = 50                  # dev rows (dev_pilot only)
K = 21                  # retrieved examples per sentence (as in the ICL runs)
STRATEGY = "all_in_one"
OUT = f"/content/drive/MyDrive/igt-word-pilot/{LANG}"
os.makedirs(OUT, exist_ok=True)
SPLIT, LIMIT, FORMATS = ("test", None, ("word",)) if MODE == "test_word" else ("dev", N, ("line", "word"))
print("results ->", OUT, "| split", SPLIT, "| formats", FORMATS)

## 4. Run both formats

Same rows, same examples, loop stop + float32 (`--fast --dtype float32`). `--resume`: if Colab disconnects, run sections 1–3 and this cell again and it continues.

In [ ]:
import sys, importlib
os.chdir(REPO)
sys.path.insert(0, f"{REPO}/icl")
import run_icl
importlib.reload(run_icl)
for fmt in FORMATS:
    print(f"===== format: {fmt} =====", flush=True)
    args = ["--lang", LANG, "--split", SPLIT, "--k", str(K), "--strategy", STRATEGY,
            "--fast", "--dtype", "float32", "--resume", "--out-dir", OUT, "--format", fmt]
    if LIMIT:
        args += ["--limit", str(LIMIT)]
    run_icl.main(args)

## 5. Compare (same scoring as the paper table: loop stop, affixes re-joined)

In [ ]:
import pandas as pd
from pathlib import Path
sys.path.insert(0, f"{REPO}/evaluation")
import rescore_loop_stop as rs
end = f"{SPLIT}_n{LIMIT}" if LIMIT else SPLIT
files = {"word": Path(f"{OUT}/bolmo-1b_{STRATEGY}_icl{K}_word_{end}.csv")}
line_file = Path(f"{OUT}/bolmo-1b_{STRATEGY}_icl{K}_{end}.csv")
if not line_file.exists():  # test_word: the existing line-format test run in the repo
    line_file = Path(f"{REPO}/results/icl/{LANG}/bolmo-1b_{STRATEGY}_icl{K}_{end}.csv")
if line_file.exists():
    files = {"line": line_file, **files}
else:
    print("No line-format file to compare with:", line_file)
rows = []
for fmt, f in files.items():
    r = [x for x in rs.rescore(LANG, STRATEGY, f) if x["rule"] == "loop_stop"][0]
    d = pd.read_csv(f, dtype=str, keep_default_na=False)
    same = (d.pred_gloss.str.split().map(len) == d.gold_gloss.str.split().map(len)).mean()
    rows.append({"format": fmt, "morpheme acc": r["gloss_morpheme_accuracy"], "word acc": r["gloss_word_accuracy"],
                 "morpheme bag F1": r["gloss_morpheme_bag_f1"], "MER": r["gloss_mer"], "gloss chrF++": r["gloss_chrfpp"],
                 "translit chrF++": r["transliteration_chrfpp"], "parse failure": r["parse_failure_rate"],
                 "loop rate": r["stopped_by_loop"], "gold word count": same})
res = pd.DataFrame(rows).set_index("format")
pd.set_option("display.width", 200)
print(res.round(3).to_string())

## 6. Download the pilot files (send the zip to Soyeong)

In [ ]:
import glob, zipfile
from google.colab import files as colab_files
end = f"{SPLIT}_n{LIMIT}" if LIMIT else SPLIT
zip_path = f"/content/word_{LANG}_{end}.zip"
with zipfile.ZipFile(zip_path, "w") as z:
    for f in glob.glob(f"{OUT}/bolmo-1b_{STRATEGY}_icl{K}*_{end}.*"):
        z.write(f, arcname=os.path.basename(f))
print(sorted(zipfile.ZipFile(zip_path).namelist()))
colab_files.download(zip_path)